# FK Benchmark: CPU vs TPU (Colab)

This notebook benchmarks forward kinematics (FK) on CPU vs TPU using JAX.
It is intended to run in Google Colab with a TPU runtime.

Notes:
- Ensure the TPU runtime is enabled (Runtime > Change runtime type > TPU).
- This uses JAX on both CPU and TPU (no CUDA).
- Data transfer to devices is excluded from timing.


In [ ]:
# --- Colab setup (uncomment if needed) ---
# If you cloned the repo in Colab, set the repo root, then install editable.
# %cd /content/pyronot
# %pip install -e .
# %pip install robot_descriptions

# JAX TPU wheels (Colab TPU).
# %pip install -q \n#   "jax[tpu]" \n#   -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import pyronot as pk
from robot_descriptions.loaders.yourdfpy import load_robot_description

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

cpu_devices = jax.devices("cpu")
tpu_devices = jax.devices("tpu")
assert cpu_devices, "No CPU device found."
assert tpu_devices, "No TPU device found. Ensure TPU runtime is enabled."
cpu = cpu_devices[0]
tpu = tpu_devices[0]
print("CPU device:", cpu)
print("TPU device:", tpu)


In [ ]:
# --- Configuration ---
ROBOT_NAME = "panda"
BATCH_SIZES = [1, 16, 64, 256, 1024, 4096, 8192, 16384]
N_WARMUP = 5
N_TIMED = 50
ATOL = 1e-4
RTOL = 1e-4

print("Loading robot description...")
urdf = load_robot_description(f"{ROBOT_NAME}_description")
robot = pk.Robot.from_urdf(urdf)
n_act = robot.joints.num_actuated_joints
print(f"{n_act} actuated joints, {robot.joints.num_joints} total joints, {robot.links.num_links} links")


In [ ]:
# --- Helpers ---
def _time_fn(fn, *args, n=N_TIMED):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        out = fn(*args)
        jax.block_until_ready(out)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

fk_cpu = jax.jit(lambda cfg: robot.forward_kinematics(cfg, use_cuda=False), backend="cpu")
fk_tpu = jax.jit(lambda cfg: robot.forward_kinematics(cfg, use_cuda=False), backend="tpu")


In [ ]:
# --- Benchmark ---
rng = np.random.default_rng(42)
lo = np.array(robot.joints.lower_limits)
hi = np.array(robot.joints.upper_limits)

print(f"\nFK performance: CPU vs TPU ({ROBOT_NAME} robot)")
print("{:<8} {:<8} {:<12} {:<12} {:<10} {}".format("Impl", "Batch", "CPU (ms)", "TPU (ms)", "Speedup", "Max |err|"))
print("-" * 70)

all_passed = True

for batch in BATCH_SIZES:
    cfg_np = rng.uniform(lo, hi, size=(batch, n_act)).astype(np.float32)
    cfg_jax = jnp.array(cfg_np)

    cfg_cpu = jax.device_put(cfg_jax, cpu)
    cfg_tpu = jax.device_put(cfg_jax, tpu)

    # Warm-up (JIT compile)
    for _ in range(N_WARMUP):
        jax.block_until_ready(fk_cpu(cfg_cpu))
        jax.block_until_ready(fk_tpu(cfg_tpu))

    # Correctness
    out_cpu = np.array(fk_cpu(cfg_cpu))
    out_tpu = np.array(fk_tpu(cfg_tpu))
    max_err = float(np.abs(out_cpu - out_tpu).max())
    passed = np.allclose(out_cpu, out_tpu, atol=ATOL, rtol=RTOL)
    all_passed &= passed

    # Timing (compute only; no host->device transfer)
    t_cpu = _time_fn(fk_cpu, cfg_cpu)
    t_tpu = _time_fn(fk_tpu, cfg_tpu)
    speedup = t_cpu / t_tpu if t_tpu > 0 else float("nan")

    status = "OK" if passed else "FAIL"
    print(f"{"CPU":<8} {batch:<8} {t_cpu*1e3:>10.3f}   {t_tpu*1e3:>10.3f}   {speedup:>7.2f}x   |err|={max_err:.2e}  [{status}]")

print("-" * 70)
if all_passed:
    print(f"PASSED: CPU and TPU outputs agree within tolerance (atol={ATOL}, rtol={RTOL}).")
else:
    print("FAILED: one or more batch sizes exceeded the tolerance threshold.")
